# Time Weighted Vector Store Retriever — 시간 가중치 검색

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- **TimeWeightedVectorStoreRetriever**의 시간 가중치 계산 방식 이해하기
- `decay_rate`로 시간 감소율을 조절하는 방법 익히기
- 뉴스, 대화 기록 등 시간에 민감한 데이터에 적용하기

## 사전 지식

- VectorStoreRetriever 기본 사용법
- 최신 정보와 오래된 정보의 관련성이 다를 수 있다는 도메인 지식

---

## 핵심 개념

### 일반 Vector Search vs Time Weighted Search

| 방식 | 점수 계산 |
|------|----------|
| 일반 검색 | 유사도만 고려 |
| **Time Weighted** | 유사도 × 시간 가중치 |

예시:
```
문서 A (1년 전, 유사도 0.9)  → 시간 감소 후: 0.6  (선택 안 됨)
문서 B (어제,   유사도 0.8)  → 시간 가중치: 0.9  (선택됨)
```

## 사용 시나리오

- 뉴스 피드 검색
- 실시간 정보 업데이트
- 대화 기록 기반 검색
- 시간에 민감한 금융·주식 데이터

> **실무 팁**: `decay_rate`를 아주 작은 값(예: 1e-24)으로 설정하면 사실상 시간 순서만 반영돼요. 반대로 0.9처럼 크게 설정하면 어제 데이터도 크게 감소합니다.

> 🔑 **핵심 개념**: 최종 점수 공식은 `score = semantic_similarity × (1 - decay_rate)^hours_passed`입니다. `decay_rate`가 높을수록 시간이 지난 문서의 점수가 빠르게 떨어져요.

> ⚠️ **자주 하는 실수**: 시간 무관한 데이터(역사적 사실, 백과사전 등)에 TimeWeighted를 적용하면 오히려 검색 품질이 떨어집니다. 최신성이 실제로 중요한 도메인에서만 사용하세요.

In [1]:
from dotenv import load_dotenv
load_dotenv()


True

In [2]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# ---------------------------------------------------
# 시간 메타데이터가 다른 문서를 추가하고 최신 문서 우선 검색 확인
# ---------------------------------------------------

# ============================================================
# TODO: TimeWeightedVectorStoreRetriever를 생성하고 시간차를 둔 문서를 추가하세요
# 힌트: decay_rate가 작을수록 시간 감소 속도가 느림 (최신 문서 우대 효과 약함)
# 예상 결과: 검색 결과에서 최신 문서가 상위에 위치
# ============================================================

from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain.retrievers import TimeWeightedVectorStoreRetriever
from langchain_core.documents import Document
from datetime import datetime, timedelta
import time

# Vectorstore 준비
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_texts(
    ["빈 초기화용"],
    embedding=embeddings
)

# TimeWeightedVectorStoreRetriever 생성
# decay_rate=0.01: 반감기 약 70일, 8시간이면 시간 점수 약 0.92
# 최종 점수 = 유사도 + (1 - decay_rate)^경과시간
time_weighted_retriever = TimeWeightedVectorStoreRetriever(
    vectorstore=vectorstore,
    decay_rate=0.01,
    k=3
)

# 시간차를 두고 문서 추가 (last_accessed_at으로 최신성 지정)
# 💡 비슷한 주제의 풍부한 문서를 사용하면 유사도가 균등해져서
#    시간 가중치의 효과를 명확하게 관찰할 수 있음
docs_with_time = [
    ("2020년 AI 기술 동향 보고서입니다. 이 해의 가장 큰 혁신은 OpenAI의 GPT-3 발표였습니다. "
     "1750억 파라미터의 대규모 언어 모델이 Few-shot 학습으로 번역, 요약, 코드 생성 등 "
     "다양한 자연어 처리 태스크에서 뛰어난 성능을 보여주었습니다.", 72),   # 72시간 전
    ("2023년 AI 기술 동향 보고서입니다. 이 해의 가장 큰 혁신은 멀티모달 AI의 부상이었습니다. "
     "GPT-4V와 Gemini 등이 텍스트와 이미지를 동시에 이해하고 생성하는 능력을 선보이며 "
     "시각적 질의응답, 문서 분석 등 새로운 활용 사례가 폭발적으로 증가했습니다.", 8),    # 8시간 전
    ("2024년 AI 기술 동향 보고서입니다. 이 해의 가장 큰 혁신은 소형 언어 모델(SLM)의 부상이었습니다. "
     "Phi-3, Gemma 등이 스마트폰에서 직접 실행 가능해지며 온디바이스 AI 시대가 열렸고 "
     "클라우드 의존도를 낮추면서도 실용적인 AI 기능을 제공합니다.", 0),                    # 현재
]

for content, hours_ago in docs_with_time:
    time.sleep(0.1)
    doc = Document(
        page_content=content,
        metadata={"last_accessed_at": datetime.now() - timedelta(hours=hours_ago)}
    )
    time_weighted_retriever.add_documents([doc])

print("✅ TimeWeightedVectorStoreRetriever 생성 완료")

✅ TimeWeightedVectorStoreRetriever 생성 완료


In [3]:
# 검색
query = "인공지능 기술 혁신"
docs = time_weighted_retriever.invoke(query)

print(f"📝 쿼리: {query}\n")
print("="*80)
print("[Time Weighted 결과]")
print("="*80)
for i, doc in enumerate(docs, 1):
    print(f"[{i}] {doc.page_content[:60]}...")
print()

print("💡 특징:")
print("  - 유사도만으로는 2023년 문서가 1위이지만, 시간 가중치가 이를 역전")
print("  - decay_rate로 시간 감소 속도 조절")
print("  - 시간에 민감한 정보 검색에 적합")

📝 쿼리: 인공지능 기술 혁신

[Time Weighted 결과]
[1] 2024년 AI 기술 동향 보고서입니다. 이 해의 가장 큰 혁신은 소형 언어 모델(SLM)의 부상이었습니다....
[2] 2023년 AI 기술 동향 보고서입니다. 이 해의 가장 큰 혁신은 멀티모달 AI의 부상이었습니다. GPT-4...
[3] 2020년 AI 기술 동향 보고서입니다. 이 해의 가장 큰 혁신은 OpenAI의 GPT-3 발표였습니다. 1...

💡 특징:
  - 유사도만으로는 2023년 문서가 1위이지만, 시간 가중치가 이를 역전
  - decay_rate로 시간 감소 속도 조절
  - 시간에 민감한 정보 검색에 적합


/Users/mhso/working/hankyung-llm-agent/02_langchain_basics/.venv/lib/python3.11/site-packages/langchain/retrievers/time_weighted_retriever.py:86: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='d1431084-19fc-4ba0-aafe-2267a2c4bf85', metadata={'last_accessed_at': datetime.datetime(2026, 4, 10, 2, 9, 56, 124851), 'created_at': datetime.datetime(2026, 4, 10, 10, 9, 56, 124965), 'buffer_idx': 1}, page_content='2023년 AI 기술 동향 보고서입니다. 이 해의 가장 큰 혁신은 멀티모달 AI의 부상이었습니다. GPT-4V와 Gemini 등이 텍스트와 이미지를 동시에 이해하고 생성하는 능력을 선보이며 시각적 질의응답, 문서 분석 등 새로운 활용 사례가 폭발적으로 증가했습니다.'), 0.3494241966652355), (Document(id='f386a5b2-a756-4d21-bc65-bc3b7eaf30b7', metadata={'last_accessed_at': datetime.datetime(2026, 4, 10, 10, 9, 56, 343114), 'created_at': datetime.datetime(2026, 4, 10, 10, 9, 56, 343214), 'buffer_idx': 2}, page_content='2024년 AI 기술 동향 보고서입니다. 이 해의 가장 큰 혁신은 소형 언어 모델(SLM)의 부상이었습니다. Phi-3, Gemma 등이 스마트폰에서 직접 실행 가능해지며 온디바이스 AI 시대가 열렸고 클라우드 의존도를 낮추면서도 실용적인 AI 기능을 제공합니다.'), 0.3176277

In [4]:
# ---------------------------------------------------
# decay_rate 변화에 따른 검색 순위 비교
# ---------------------------------------------------
import time

decay_rates = [
    (0.0000000000000000000000001, "극소 (1e-25) — 시간 무관"),
    (0.01, "낮음 (0.01) — 반감기 약 70일"),
    (0.5, "높음 (0.5) — 반감기 약 1.4일"),
]

# 동일 문서 세트로 각 decay_rate별 retriever 생성 및 비교
compare_query = "인공지능 기술 혁신"

print(f"📝 쿼리: {compare_query}")
print(f"문서: 72시간 전(2020년) / 8시간 전(2023년) / 현재(2024년)\n")
print(f"{'='*78}")
print(f"{'순위':<5} {'극소 (시간 무관)':<27} {'낮음 (0.01)':<27} {'높음 (0.5)':<27}")
print(f"{'='*78}")

all_results = {}
for rate, label in decay_rates:
    vs = FAISS.from_texts(["init"], embedding=embeddings)
    twvr = TimeWeightedVectorStoreRetriever(
        vectorstore=vs, decay_rate=rate, k=3
    )
    for content, hours_ago in docs_with_time:
        time.sleep(0.05)
        d = Document(
            page_content=content,
            metadata={"last_accessed_at": datetime.now() - timedelta(hours=hours_ago)}
        )
        twvr.add_documents([d])
    all_results[label] = twvr.invoke(compare_query)

# 순위 비교 테이블
hours_map = {"2020": "72h전", "2023": "8h전", "2024": "현재"}
for rank in range(3):
    row = []
    for _, label in decay_rates:
        results = all_results[label]
        if rank < len(results):
            content = results[rank].page_content
            year = content[:4]
            tag = hours_map.get(year, "")
            row.append(f"{year}년 ({tag})")
        else:
            row.append("-")
    print(f"[{rank+1}]  {row[0]:<27} {row[1]:<27} {row[2]:<27}")

print(f"\n💡 관찰 포인트:")
print("  - 극소 (1e-25): 시간 효과 없음 → 유사도 순서대로 (2023 > 2024 > 2020)")
print("  - 낮음 (0.01):  시간 가중치가 유사도 차이를 역전 → 최신 문서(2024) 우선!")
print("  - 높음 (0.5):   8시간 전도 급격히 하락 → 사실상 최신만 의미 있음")

📝 쿼리: 인공지능 기술 혁신
문서: 72시간 전(2020년) / 8시간 전(2023년) / 현재(2024년)

순위    극소 (시간 무관)                  낮음 (0.01)                   높음 (0.5)                   


/Users/mhso/working/hankyung-llm-agent/02_langchain_basics/.venv/lib/python3.11/site-packages/langchain/retrievers/time_weighted_retriever.py:86: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='aeef9d73-95b0-46da-a167-d810c11028f6', metadata={'last_accessed_at': datetime.datetime(2026, 4, 10, 2, 9, 57, 224791), 'created_at': datetime.datetime(2026, 4, 10, 10, 9, 57, 224858), 'buffer_idx': 1}, page_content='2023년 AI 기술 동향 보고서입니다. 이 해의 가장 큰 혁신은 멀티모달 AI의 부상이었습니다. GPT-4V와 Gemini 등이 텍스트와 이미지를 동시에 이해하고 생성하는 능력을 선보이며 시각적 질의응답, 문서 분석 등 새로운 활용 사례가 폭발적으로 증가했습니다.'), 0.34918067217453963), (Document(id='67bf5720-6073-4c97-a1c4-b2fdca42b99e', metadata={'last_accessed_at': datetime.datetime(2026, 4, 10, 10, 9, 57, 850882), 'created_at': datetime.datetime(2026, 4, 10, 10, 9, 57, 850923), 'buffer_idx': 2}, page_content='2024년 AI 기술 동향 보고서입니다. 이 해의 가장 큰 혁신은 소형 언어 모델(SLM)의 부상이었습니다. Phi-3, Gemma 등이 스마트폰에서 직접 실행 가능해지며 온디바이스 AI 시대가 열렸고 클라우드 의존도를 낮추면서도 실용적인 AI 기능을 제공합니다.'), 0.317190

[1]  2023년 (8h전)                 2024년 (현재)                  2024년 (현재)                 
[2]  2024년 (현재)                  2023년 (8h전)                 2023년 (8h전)                
[3]  2020년 (72h전)                2020년 (72h전)                2020년 (72h전)               

💡 관찰 포인트:
  - 극소 (1e-25): 시간 효과 없음 → 유사도 순서대로 (2023 > 2024 > 2020)
  - 낮음 (0.01):  시간 가중치가 유사도 차이를 역전 → 최신 문서(2024) 우선!
  - 높음 (0.5):   8시간 전도 급격히 하락 → 사실상 최신만 의미 있음


/Users/mhso/working/hankyung-llm-agent/02_langchain_basics/.venv/lib/python3.11/site-packages/langchain/retrievers/time_weighted_retriever.py:86: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='62a08be3-8349-4004-a431-d98ad95d174b', metadata={'last_accessed_at': datetime.datetime(2026, 4, 10, 2, 10, 0, 195919), 'created_at': datetime.datetime(2026, 4, 10, 10, 10, 0, 195972), 'buffer_idx': 1}, page_content='2023년 AI 기술 동향 보고서입니다. 이 해의 가장 큰 혁신은 멀티모달 AI의 부상이었습니다. GPT-4V와 Gemini 등이 텍스트와 이미지를 동시에 이해하고 생성하는 능력을 선보이며 시각적 질의응답, 문서 분석 등 새로운 활용 사례가 폭발적으로 증가했습니다.'), 0.3525066064310798), (Document(id='a369560c-910a-4214-841a-6419c6b0c117', metadata={'last_accessed_at': datetime.datetime(2026, 4, 10, 10, 10, 0, 499142), 'created_at': datetime.datetime(2026, 4, 10, 10, 10, 0, 499290), 'buffer_idx': 2}, page_content='2024년 AI 기술 동향 보고서입니다. 이 해의 가장 큰 혁신은 소형 언어 모델(SLM)의 부상이었습니다. Phi-3, Gemma 등이 스마트폰에서 직접 실행 가능해지며 온디바이스 AI 시대가 열렸고 클라우드 의존도를 낮추면서도 실용적인 AI 기능을 제공합니다.'), 0.3176027

---

## 마무리 정리

### 핵심 요약

| 항목 | 내용 |
|------|------|
| 핵심 아이디어 | 최종 점수 = 의미 유사도 × (1 - decay_rate)^경과시간(시간 단위) |
| `decay_rate` 역할 | 값이 클수록 오래된 문서 점수가 빠르게 낮아짐 |
| 문서 접근 갱신 | 검색될 때마다 `last_accessed_at` 자동 갱신 |
| 적합한 용도 | 뉴스, 실시간 데이터, 날짜 민감한 FAQ |
| 주의 | 날짜 무관한 정보엔 비적합 — 유사도만 순수하게 쓰는 게 나을 수 있음 |

### `decay_rate` 선택 가이드

| `decay_rate` | 반감 속도 | 적합한 상황 |
|--------------|-----------|-------------|
| 0.01 | 약 70일 | 월간 업데이트 문서 |
| 0.1 | 약 7일 | 주간 뉴스, 블로그 |
| 0.5 | 약 1.4일 | 실시간 이슈 Q&A |
| 0.99 | 거의 즉시 | 당일 데이터만 유효한 경우 |

### 다음 노트북 예고

**10-Kiwi-BM25Retriever** — 한국어 형태소 분석기 Kiwi를 BM25와 결합해 한국어 키워드 검색 정확도를 높이는 방법을 배워요.